In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [3]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass #paramatorはない

    def fit(self,X,y):
        #ここで渡されるXは、学習データ（train_X）のみ
        #統計量(medianなど)を使う場合はここで定義しないとデータリークにつながる
        X = X.copy()
        y = y.copy()
        
        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace(["Mlle", "Ms"], "Miss")
        X["Title"] = X["Title"].replace("Mme", "Mrs")
        rare_titles = [
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X["Title"] = X["Title"].replace(rare_titles, "Rare")
        
        self.title = X["Title"]
        self.title_age_median = X.groupby("Title")["Age"].median()
        self.global_age_median = X["Age"].median()

        #"familygroup"用の統計量をfit時のみ計算
        #統計つくりのための特徴量作り的な
        surname = X["Name"].str.extract(r"^([^,]+),",expand=False)
        family = X["SibSp"] + X["Parch"]

        dummy_df = pd.DataFrame({
            "Surname":surname,
            "family":family,
            "Survived":y #fitの時だけ参照、データリーク防止
        })

        #苗字グループの生存率と同苗字数
        self.surname_surv_stats = (
            dummy_df.groupby("Surname")["Survived"].agg(
                surv_rate = "mean",
                group_count = "count"
            )
        )

        #学習用データでの苗字の出現回数
        self.surname_counts_in_trains = (
            dummy_df["Surname"].value_counts()
        )

        return self
        
    def transform(self,X): #ここから新特徴量の作成
        X_new = X.copy() 
        #渡されたXによって基準が変わるのでこれを使った統計量を使うと
        #データリークをしてしまう点に注意
        #つまり、transform()内で統計量は作った処理を書くな
        #渡されるXは、学習データだったり、検証データだったりする

        #"Sex_Pclass"の作成 
        X_new["Sex_Pclass"] = (
            X_new["Sex"].astype(str) + "_" +
            X_new["Pclass"].astype(str)
        )

        #"len_fam"の作成
        X_new["family"] = X_new["Parch"].astype("Int64") + X_new["SibSp"].astype("Int64")
        X_new["len_fam"] = X_new["family"].apply(
            lambda x: (
                0 if x == 0 else
                1 if 1<=x<=3 else
                2 
            )
        )

        #"logFare"の作成
        fare_per_person = X_new["Fare"] / (X_new["family"]+1)
        X_new["logFare"] = np.log1p(fare_per_person)
        

        #"Title"の作成
        #X_new["Title"] = X_new["Name"].str.extract(r" ([A-Za-z]+)\.",expand = False)
        #X_new["Title"] = X_new["Title"].replace(["Mlle","Ms"],"Miss")
        #X_new["Title"] = X_new["Title"].replace("Mme","Mrs")
        #rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
        #X_new["Title"] = X_new["Title"].replace(rare_titles,"Rare")
        X_new["Title"] = self.title
        
        #”Age"の再定義。年齢の欠損を敬称別の中央値の年齢で埋める
        #train用のデータで学習した中央値で代入。また、validに未知のTitleがある場合は、全体平均を使用
        X_new["Age"] = X_new["Age"].fillna(X_new["Title"].map(self.title_age_median))
        X_new["Age"] = X_new["Age"].fillna(self.global_age_median)
        #Ageのビニング
        age_child = (X_new["Age"]<12)
        age_young = (X_new["Age"]>=12) & (X_new["Age"]<20)
        age_adult = (X_new["Age"]>=20) & (X_new["Age"]<60)
        age_senior = (X_new["Age"]>=60)
        age_selection = [age_child,age_young,age_adult,age_senior]
        name = ["child","young","adult","senior"]
        X_new["age_binning"] = np.select(age_selection,name, default ="unknown" )

        #データリークのため、この特徴量を削除
        #追記：データリークではなかったが、年齢による線引きを変更
        #"surv_rank"を作成、デフォルトで0点上記の条件以外の部分
        condition_2points = ( #子供最優先
            ((X_new["Sex"]=="male") & (X_new["Age"]<12)) |
            ((X_new["Sex"]=="female") & ((X_new["Age"]<12) ))
        )
        condition_1point = ( #次に全女性優先
            (X_new["Sex"]=="female") & (X_new["Age"]>=12)
        )
        
        conditions = [condition_2points,condition_1point]
        points = [2,1]
        X_new["surv_rank"] = np.select(conditions,points,default=0)

        #"has_cabin"を作成
        X_new["has_cabin"] = X_new["Cabin"].notnull().astype(int)

        #"cabin_deck"を作成
        X_new["cabin_deck"] = X_new["Cabin"].str[0].fillna("Unknown")
        X_new.loc[X_new["cabin_deck"].isin(["G","T"]),"cabin_deck"] = "Unknown"

        #"Surname"の作成
        X_new["Surname"]= X_new["Name"].str.extract(r"^([^,]+),", expand=False)

        #"familygroup"の作成
        def assign_family_group(row):
            #transform側に渡されたdf(つまり新たなtrainとtest)を1行ずつ(row)読み込む
            surname = row["Surname"]
            
            if row["family"] == 0:
                return "alone"

            if pd.isna(surname):
                return "mixed"

            if surname not in self.surname_surv_stats.index:
                return "mixed"

            if self.surname_surv_stats.at[surname, "group_count"] == 1:
                return "mixed"

            #イメージとしてはsurnameは今手渡しされた情報、surv_rateは事前に混ざらぬよう渡された情報
            surv_rate = self.surname_surv_stats.loc[surname,"surv_rate"]

            if surv_rate ==0.0:
                return "family_all_died"
            elif surv_rate == 1.0:
                return "family_all_survived"
            else:
                return "family_mixed" #一部生存、一部死亡

        X_new["FamilyGroup"] = X_new.apply(assign_family_group,axis=1)

        #使う列の指定
        num_cols = ["Age","Pclass","logFare",]
        cat_cols = ["Sex","FamilyGroup","has_cabin","Embarked"]

        X_new[num_cols] = X_new[num_cols].astype(float)
        X_new[cat_cols] = X_new[cat_cols].astype("category")
        final_features = num_cols + cat_cols
        
        
        return X_new[final_features]

# Pipeline作成
前回はこちらが用意した前処理方法をPipelineに組み込んでいたが、今回はモデル事態に全て任せる手法をとる。

In [5]:
#cros_val_score時の分割時の詳細設定
from sklearn.model_selection import StratifiedKFold

cv =StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

In [6]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING) #途中式を消すため
from xgboost import XGBClassifier
from sklearn import set_config #cross_val_scoreを後で使えるように

set_config(transform_output="pandas") #feature_engineeringから返るSeriesをpandasへ

XGBC_pipeline = Pipeline(steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("model",XGBClassifier(
            n_jobs=-1,
            random_state=10,
            eval_metric = "logloss",#警告文を消すため
            learning_rate = 0.05,
            n_estimators =300,
            enable_categorical=True, #カテゴリ処理のため
        ))
    ]
)

In [7]:
#パラメータのチューニングに伴い、新たなXGBClassifierのモデルコード作成

def XGBC_func(trial):
    params = {
        "colsample_bytree":trial.suggest_float("colsample_bytree",0.6,0.9),
        "reg_alpha":trial.suggest_float("reg_alpha",1e-2,5.0,log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 5.0, log=True),
        "max_depth":trial.suggest_int("max_depth",3,10),
        "min_child_weight":trial.suggest_int("min_child_weight",1,8),
        "subsample":trial.suggest_float("subsample",0.6,0.9)
    }

    #Pipelineへのパラメータ探索のセット
    XGBC_pipeline.set_params(**{f"model__{k}":v for k,v in params.items()} )

    scores = cross_val_score(XGBC_pipeline,X,y,cv=cv,scoring="accuracy")

    return scores.mean()

study = optuna.create_study(direction = "maximize")
study.optimize(XGBC_func,n_trials=50,show_progress_bar=True)

print(f"ベストスコア:{study.best_value}")
print(f"ベストパラメータ:{study.best_params}")

XGBC_best_value=study.best_value
XGBC_best_params = study.best_params

  0%|          | 0/50 [00:00<?, ?it/s]

ベストスコア:0.8238152030632101
ベストパラメータ:{'colsample_bytree': 0.6196547423742743, 'reg_alpha': 0.23947834131361065, 'reg_lambda': 2.2692058005797273, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.7933234272810926}


In [8]:
XGBC_pipeline.set_params(
    **{f"model__{k}":v for k,v in XGBC_best_params.items()}
)
    
XGBC_pipeline.fit(X,y)

preds = XGBC_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)

03_eda&FeatureEngineeringの追記より、新しい特徴量として"familygroup"を作成する

# スコア改善ログ
- score:0.770 cv=0.855
  >使った特徴量
  >
  >num_cols = ["Age","Pclass","family","logFare","surv_rank","has_cabin"]
  >
  >cat_cols = ["age_binning","Sex","Title","Embarked","cabin_deck","Sex_Pclass"]
  >
* score:0.785 cv=0.845
  >num_cols = ["Pclass","family","logFare","has_cabin"]
  >
   >     cat_cols = ["age_binning","Sex","Embarked","cabin_deck"]
  
  >
  >特徴量の選別というより、パラメータが過学習を起こしていることに気づく。max_depth=8は、Titanicのデータ数に対して深すぎる。こちら側で探索するパラメータの範囲を制限
* score:0.780 cv=0.854
  >        num_cols = ["Age","logFare","has_cabin","surv_rank"]
  > 
    >    cat_cols = ["Pclass","Sex","Title","Embarked","cabin_deck"]
* 「03_eda&FeatureEngineering」追記より、新しい特徴量として"FamilyGroup"を追加する
  >特徴量の選定だけではスコアが伸びなかったため
* score:0.775 cv=0.831
  >num_cols = ["Age","logFare"]

  > cat_cols = ["Sex","Pclass","Title","Embarked","cabin_deck","has_cabin","FamilyGroup"]

In [9]:
output

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0
